In [1]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

# A1. Bronze->silver orders dedupe by order_id latest ingest_ts.

- Source
  ↓
- Bronze
  ↓
- Silver
  ↓
- Gold

In [8]:
bronze_orders = spark.createDataFrame(
    [
        ("o101", "c1", 100, "2024-01-01 10:00:00"),
        ("o101", "c1", 120, "2024-01-01 11:00:00"),
        ("o102", "c2", 200, "2024-01-01 10:30:00"),
        ("o103", "c3", 300, "2024-01-01 09:00:00"),
        ("o103", "c3", 350, "2024-01-01 12:00:00"),
    ],
    ["order_id", "customer_id", "amount", "ingest_ts"]
)
bronze_orders.show()

+--------+-----------+------+-------------------+
|order_id|customer_id|amount|          ingest_ts|
+--------+-----------+------+-------------------+
|    o101|         c1|   100|2024-01-01 10:00:00|
|    o101|         c1|   120|2024-01-01 11:00:00|
|    o102|         c2|   200|2024-01-01 10:30:00|
|    o103|         c3|   300|2024-01-01 09:00:00|
|    o103|         c3|   350|2024-01-01 12:00:00|
+--------+-----------+------+-------------------+



In [9]:
bronze_orders = bronze_orders.withColumn(
    "ingest_ts",
    F.to_timestamp("ingest_ts")
)

In [13]:
dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("ingest_ts").desc())
)


In [11]:
silver_orders = (
    bronze_orders
    .withColumn(
        "rn",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)
silver_orders.show()

+--------+-----------+------+-------------------+
|order_id|customer_id|amount|          ingest_ts|
+--------+-----------+------+-------------------+
|    o101|         c1|   120|2024-01-01 11:00:00|
|    o102|         c2|   200|2024-01-01 10:30:00|
|    o103|         c3|   350|2024-01-01 12:00:00|
+--------+-----------+------+-------------------+



# A2. Gold daily revenue by country.

In [18]:
silver_orders = spark.createDataFrame(
    [
        ("o101", "US", "2024-01-01", 120),
        ("o102", "US", "2024-01-01", 200),
        ("o103", "UK", "2024-01-01", 350),
        ("o104", "UK", "2024-01-02", 150),
        ("o105", "India", "2024-01-02", 500),
    ],
    ["order_id", "country", "order_date", "amount"]
)
silver_orders.show()

+--------+-------+----------+------+
|order_id|country|order_date|amount|
+--------+-------+----------+------+
|    o101|     US|2024-01-01|   120|
|    o102|     US|2024-01-01|   200|
|    o103|     UK|2024-01-01|   350|
|    o104|     UK|2024-01-02|   150|
|    o105|  India|2024-01-02|   500|
+--------+-------+----------+------+



In [16]:
silver_orders = silver_orders.withColumn(
    "order_date",
    F.to_date("order_date")
)

In [19]:
gold_daily_revenue = (
    silver_orders
    .groupBy(
        "order_date",
        "country"
    )
    .agg(
        F.sum("amount").alias("daily_revenue"),
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy(
        "order_date",
        "country"
    )
)
gold_daily_revenue.show()

+----------+-------+-------------+-----------+
|order_date|country|daily_revenue|order_count|
+----------+-------+-------------+-----------+
|2024-01-01|     UK|          350|          1|
|2024-01-01|     US|          320|          2|
|2024-01-02|  India|          500|          1|
|2024-01-02|     UK|          150|          1|
+----------+-------+-------------+-----------+



# A3. Build SCD2 sample history DataFrame for city changes.
                Source
                   |
                   v
            Customer exists?
                   |
             +-----+-----+
             |           |
            No          Yes
             |           |
             v           v
          INSERT     City changed?
                         |
                    +----+----+
                    |         |
                   No        Yes
                    |         |
                    v         v
                  Ignore   Expire old
                              |
                              v
                         Insert new

In [23]:
scd2_customer_history = spark.createDataFrame(
    [
        (
            "c1",
            "Rahul",
            "Bangalore",
            "2024-01-01",
            "2024-06-30",
            False
        ),
        (
            "c1",
            "Rahul",
            "Hyderabad",
            "2024-07-01",
            "9999-12-31",
            True
        ),
        (
            "c2",
            "Amit",
            "Mumbai",
            "2024-01-01",
            "9999-12-31",
            True
        ),
    ],
    [
        "customer_id",
        "customer_name",
        "city",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    ]
)
scd2_customer_history.show()

+-----------+-------------+---------+--------------------+------------------+----------+
|customer_id|customer_name|     city|effective_start_date|effective_end_date|is_current|
+-----------+-------------+---------+--------------------+------------------+----------+
|         c1|        Rahul|Bangalore|          2024-01-01|        2024-06-30|     false|
|         c1|        Rahul|Hyderabad|          2024-07-01|        9999-12-31|      true|
|         c2|         Amit|   Mumbai|          2024-01-01|        9999-12-31|      true|
+-----------+-------------+---------+--------------------+------------------+----------+



In [22]:
# convert to date type
scd2_customer_history = (
    scd2_customer_history
    .withColumn(
        "effective_start_date",
        F.to_date("effective_start_date")
    )
    .withColumn(
        "effective_end_date",
        F.to_date("effective_end_date")
    )
)
scd2_customer_history.show()

+-----------+-------------+---------+--------------------+------------------+----------+
|customer_id|customer_name|     city|effective_start_date|effective_end_date|is_current|
+-----------+-------------+---------+--------------------+------------------+----------+
|         c1|        Rahul|Bangalore|          2024-01-01|        2024-06-30|     false|
|         c1|        Rahul|Hyderabad|          2024-07-01|        9999-12-31|      true|
|         c2|         Amit|   Mumbai|          2024-01-01|        9999-12-31|      true|
+-----------+-------------+---------+--------------------+------------------+----------+



# A4. Batch vs streaming use cases.

Event
 ↓
Kafka/Event Hub
 ↓
Spark Structured Streaming
 ↓
Delta
 ↓
Dashboard/Alert

| Feature     | Batch                 | Streaming              |
| ----------- | --------------------- | ---------------------- |
| Processing  | Periodic              | Continuous/incremental |
| Latency     | Minutes/hours/days    | Seconds/minutes        |
| Complexity  | Lower                 | Higher                 |
| Example     | Daily report          | Fraud detection        |
| Data volume | Large historical sets | Continuous events      |
| Trigger     | Schedule              | Event/data arrival     |

# **I use batch when the business can tolerate scheduled processing, such as daily revenue or monthly reporting. I use streaming when low latency is required, such as IoT monitoring, fraud detection, clickstream processing, or real-time alerts.**

# A5. How to make loads idempotent?

# I use a stable business key, deduplicate the source, and use Delta MERGE instead of append-only writes. For incremental pipelines I use watermarks, and for streaming I use checkpoints. This ensures rerunning the same data doesn't create duplicates.

                  SOURCE
                    │
          ┌─────────┴─────────┐
          │                   │
        Batch              Streaming
          │                   │
          v                   v
       Bronze              Bronze
          │                   │
          └─────────┬─────────┘
                    │
                    v
              SILVER LAYER
          Clean + Validate
          Deduplicate
          Standardize
                    │
                    v
                GOLD LAYER
          Business Aggregations
          Daily Revenue
          KPIs / Metrics
                    │
                    v
               Power BI

# And for the customer dimension:

                 Customer Source
                        │
                        v
                  Compare City
                        │
              ┌─────────┴─────────┐
              │                   │
          No Change            Changed
              │                   │
              v                   v
            Ignore          Expire Old Row
                                  │
                                  v
                           Insert New Row
                                  │
                                  v
                               SCD Type 2